# [Chapter 9 — Practical Issues in Fitting, §9.5] Pitfall 5: Correcting for Reporting Delays and Under-ascertainment

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 9 — Practical Issues in Fitting
**Considerations developed:** 9
**Estimated runtime:** ≤ 30s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Demonstrates the multiplier model: observed cases are typically a fraction $\rho$ of true cases, possibly time-varying, and reported with a delay distribution. Shows the bias when these are ignored.

## What you should already know
Chapter 8 fitting notebooks.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance
set_seed_chapter_08()
book_style()


In [ ]:
params = baseline_chapter_08()
true_R0 = params['c_I'] * params['beta'] * params['tau_R']
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']
J_true = params['c_I'] * params['beta'] * (S/params['N']) * I

# Reporting model: J_obs = rho * (J_true convolved with delay distribution)
rho = 0.4   # 40% ascertainment
delay_mean = 5  # 5-day reporting delay
delay_dist = np.exp(-(np.arange(30) - delay_mean)**2 / 8.0)
delay_dist /= delay_dist.sum()
J_observed = np.convolve(rho * J_true, delay_dist, mode='full')[:len(t)]

# Naive estimate (ignoring delay + ascertainment)
mask = (t > 15) & (t < 35)
alpha_hat_naive = J_observed[mask].mean() / I[mask].mean()
R0_naive = alpha_hat_naive * params['tau_R'] / (S[mask].mean() / params['N'])

# Corrected estimate (deconvolving + scaling by 1/rho)
J_corrected = J_observed / rho   # correct for ascertainment
# (delay correction would require more sophisticated method; the bias here is from rho alone)
alpha_hat_corrected = J_corrected[mask].mean() / I[mask].mean()
R0_corrected = alpha_hat_corrected * params['tau_R'] / (S[mask].mean() / params['N'])

print(f"True R_0: {true_R0:.3f}")
print(f"Naive R_0 (ignores rho={rho}): {R0_naive:.3f}  [biased by factor of rho]")
print(f"Corrected R_0 (rescaled by 1/rho): {R0_corrected:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, J_true, color=BOOK_COLORS['infectious'], lw=1.5, label='True J(t)')
ax.plot(t, J_observed, color=BOOK_COLORS['lambda_hat'], lw=1.5, label=f'Observed (rho={rho}, delayed)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Incidence')
ax.set_title('Pitfall 5: reporting delays + under-ascertainment')
ax.legend()
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()
